# Phase 2 — Fine-tune Qwen3-8B (LoRA bf16) trên VLSP DateArith VI, sau đó đánh giá lại

Notebook này thực hiện 2 bước:
1. **TRAIN**: SFT bằng LoRA (r=16, bf16) trên rows 1500–2999 của `date_training_dataset.txt` (1500 rows), chia 90/10. Adapter lưu local `/content/checkpoints/lora_vlsp_date_bf16/` (mất khi Colab disconnect — đúng lựa chọn của user).
2. **EVAL**: chạy lại 3 phương pháp (`zero_shot`, `few_shot k=3`, `dynamic_few_shot k=3`) trên 2 dataset (`vlsp_date` in-domain, `vlsp_duration` cross-task) với adapter → 6 run.

Output FT được ghi vào `outputs_ft/` (tách hẳn với `outputs/` baseline) để dễ so sánh.

⚠️ **Data leakage warning** với `dynamic_few_shot`: pool retrieval = rows 1500–2999 — trùng với tập train. Kết quả dynamic sau FT sẽ bias upward, ghi nhớ khi report.

## Setup

In [ ]:
# === SETUP 1 — cài môi trường (thêm peft / trl / datasets cho training) ===
!pip install -q -U transformers accelerate scikit-learn pyyaml sentence-transformers faiss-cpu
!pip install -q -U peft trl datasets
!pip uninstall -q -y torchao
# ⚠️ Sau khi chạy cell này: Runtime → Restart session rồi chạy tiếp từ SETUP 2.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.6 MB/s eta 0:00:00


In [ ]:
# === SETUP 2 — tùy chọn mount Drive + clone repo ===
import os

USE_DRIVE = False  # True: mount Drive để dùng Dataset/output trên Drive
REPO_URL = 'https://github.com/duclld1709/Temporal-Reasoning-Research.git'  # TODO: đổi
REPO_DIR = '/content/Temporal_Reasoning'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
print('CWD:', os.getcwd())
print('USE_DRIVE:', USE_DRIVE)

Cloning into '/content/Temporal_Reasoning'...
remote: Enumerating objects: 200, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 200 (delta 77), reused 177 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (200/200), 1.29 MiB | 9.82 MiB/s, done.
Resolving deltas: 100% (77/77), done.
CWD: /content/Temporal_Reasoning
USE_DRIVE: False


In [ ]:
# === SETUP 3 — paths: Dataset, output FT, checkpoint local ===
import os

if USE_DRIVE:
    DRIVE_OUT_FT = '/content/drive/MyDrive/Temporal_Reasoning/outputs_ft'
    DATASET_ROOT = '/content/drive/MyDrive/Temporal_Reasoning/Dataset'
    os.makedirs(DRIVE_OUT_FT, exist_ok=True)
    local_ds = os.path.join(REPO_DIR, 'Dataset')
    if not os.path.exists(local_ds):
        os.symlink(DATASET_ROOT, local_ds)
    print('Dataset ->', os.readlink(local_ds) if os.path.islink(local_ds) else local_ds)
else:
    DRIVE_OUT_FT = '/content/outputs_ft'
    DATASET_ROOT = os.path.join(REPO_DIR, 'Dataset')
    os.makedirs(DRIVE_OUT_FT, exist_ok=True)
    if not os.path.exists(DATASET_ROOT):
        raise FileNotFoundError(f'Không tìm thấy Dataset tại {DATASET_ROOT}.')
    print('Dataset ->', DATASET_ROOT)

# Checkpoint local Colab session (mất khi disconnect — đúng lựa chọn user).
CHECKPOINT_DIR = '/content/checkpoints/lora_vlsp_date_bf16'
os.makedirs(os.path.dirname(CHECKPOINT_DIR), exist_ok=True)

print('DRIVE_OUT_FT :', DRIVE_OUT_FT)
print('CHECKPOINT   :', CHECKPOINT_DIR)

Dataset -> /content/Temporal_Reasoning/Dataset
DRIVE_OUT_FT : /content/outputs_ft
CHECKPOINT   : /content/checkpoints/lora_vlsp_date_bf16


In [ ]:
# === SETUP 4 — preprocess raw → JSONL (Dataset/Preprocessed/) ===
!python -m src.data.preprocess

[udst_duration] 1500 samples -> Dataset/Preprocessed/udst_duration.jsonl (max_samples=1500, src=Dataset/Raw/English/UDST-DurationQA/data/test.tsv)
[bigbench_date] 369 samples -> Dataset/Preprocessed/bigbench_date.jsonl (max_samples=None, src=Dataset/Raw/English/BigBench_DateUnderstanding/task.json)
[vlsp_date] 1500 samples -> Dataset/Preprocessed/vlsp_date.jsonl (max_samples=1500, src=Dataset/Raw/Vietnamese/VLSP 2025 ViTempQA (DateArith + DurationQA) Task/TrainingDataset/date_train_dataset/date_training_dataset.txt)
[vlsp_duration] 1500 samples -> Dataset/Preprocessed/vlsp_duration.jsonl (max_samples=1500, src=Dataset/Raw/Vietnamese/VLSP 2025 ViTempQA (DateArith + DurationQA) Task/TrainingDataset/durationQA_train_dataset/duration_training_dataset.txt)


## TRAIN — SFT LoRA bf16

Thời gian dự kiến trên A100 (40GB): ~10–15 phút (3 epoch × 1350/8 ≈ 506 steps).

Smoke quick (nếu muốn test pipeline): set `cfg.num_epochs=1, cfg.train_pool_size=50` trước khi train_sft.

In [ ]:
# === TRAIN — chạy SFT LoRA, save adapter vào CHECKPOINT_DIR ===
import yaml
from src.training.sft import SFTRunConfig, train_sft

with open('configs/sft_vlsp_date_lora_bf16.yaml', encoding='utf-8') as f:
    raw = yaml.safe_load(f)
raw['output_dir'] = CHECKPOINT_DIR  # override về path local Colab
cfg = SFTRunConfig(**raw)
print('SFT config:', cfg)

ADAPTER_PATH = train_sft(cfg)
print('Adapter saved to:', ADAPTER_PATH)
!ls -lah $ADAPTER_PATH

SFT config: SFTRunConfig(model_name='Qwen/Qwen3-8B', dtype='bfloat16', trust_remote_code=True, dataset='vlsp_date', dataset_path=None, train_pool_start=1500, train_pool_size=1500, val_ratio=0.1, shuffle_seed=42, output_dir='/content/checkpoints/lora_vlsp_date_bf16', lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], num_epochs=3, per_device_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, lr_scheduler='cosine', weight_decay=0.0, max_seq_length=256, bf16=True, logging_steps=10, save_strategy='epoch', eval_strategy='epoch', save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, report_to='none', seed=42)
[sft] pool=1500 -> train=1350 val=150


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[sft] response_template (loss-mask boundary) = '<|im_start|>assistant\n'


Epoch,Training Loss,Validation Loss
1,0.004920,0.031751
2,0.000476,0.002374
3,0.000315,0.002594


[sft] adapter saved to /content/checkpoints/lora_vlsp_date_bf16
Adapter saved to: /content/checkpoints/lora_vlsp_date_bf16
total 178M
drwxr-xr-x 4 root root 4.0K Apr 30 02:25 .
drwxr-xr-x 3 root root 4.0K Apr 30 01:40 ..
-rw-r--r-- 1 root root 1.1K Apr 30 02:25 adapter_config.json
-rw-r--r-- 1 root root 167M Apr 30 02:25 adapter_model.safetensors
-rw-r--r-- 1 root root 4.1K Apr 30 02:25 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Apr 30 02:10 checkpoint-338
drwxr-xr-x 2 root root 4.0K Apr 30 02:25 checkpoint-507
-rw-r--r-- 1 root root 5.1K Apr 30 02:25 README.md
-rw-r--r-- 1 root root  694 Apr 30 02:25 tokenizer_config.json
-rw-r--r-- 1 root root  11M Apr 30 02:25 tokenizer.json


## EVAL — chạy lại 3 phương pháp × 2 dataset với adapter

Load model (base + adapter) **một lần** để reuse cho 6 run — tiết kiệm thời gian.
Output FT ghi vào `outputs_ft/`, summary.csv riêng → dễ diễn tả baseline vs FT.

In [ ]:
# === EVAL setup — load Qwen3-8B + adapter 1 lần ===
from src.models.qwen import QwenChatLM, QwenConfig
from src.runner import load_config, run

MODEL_FT = QwenChatLM(QwenConfig(
    model_name='Qwen/Qwen3-8B',
    dtype='bfloat16',
    adapter_path=ADAPTER_PATH,
))
MODEL_FT.load()
print('Model + adapter ready:', MODEL_FT.config.model_name, '+', MODEL_FT.config.adapter_path)

def run_exp_ft(cfg_path, *, verbose=True, verbose_first_n=5, verbose_every=200,
               running_score_every=100, output_dir=DRIVE_OUT_FT):
    """Helper: load config, suffix experiment_name='_ft', trỏ về outputs_ft/, chạy với MODEL_FT."""
    cfg = load_config(cfg_path)
    cfg.output_dir = output_dir
    cfg.experiment_name = cfg.experiment_name + '_ft'
    cfg.adapter_path = ADAPTER_PATH  # log vào summary.csv
    cfg.verbose = verbose
    cfg.verbose_first_n = verbose_first_n
    cfg.verbose_every = verbose_every
    cfg.running_score_every = running_score_every
    return run(cfg, model=MODEL_FT)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model + adapter ready: Qwen/Qwen3-8B + /content/checkpoints/lora_vlsp_date_bf16


### `vlsp_date` (in-domain) — kỳ vọng accuracy tăng rõ rệt so với baseline (zero=26.53%, few=36.53%, dynamic=55.6%).

In [ ]:
# === EXP 1/6 — zero_shot × vlsp_date (FT) ===
m = run_exp_ft('configs/zero_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

[runner] experiment=zero_shot_vlsp_date_ft method=zero_shot dataset=vlsp_date verbose=True
[runner] loaded 1500 samples (task=date_arith, lang=vi)
[runner] reusing pre-loaded model: Qwen/Qwen3-8B (+adapter=/content/checkpoints/lora_vlsp_date_bf16)
  [1/1500] ✓ gold='Tháng 4, 1321' extracted='Tháng 4, 1321' elapsed=1.33s
      Q: Giả sử bạn đang ở tháng 4, 1316, thời gian sau 4 năm 12 tháng, thì là thời điểm nào?
      raw: Tháng 4, 1321
  [2/1500] ✓ gold='Tháng 2, 1078' extracted='Tháng 2, 1078' elapsed=1.06s
      Q: Hãy tính thời điểm 10 năm trước tháng 2, 1088
      raw: Tháng 2, 1078
  [3/1500] ✓ gold='Tháng 1, 1478' extracted='Tháng 1, 1478' elapsed=1.06s
      Q: Ngày tháng nào sẽ là 1 năm 10 tháng trước tháng 11, 1479?
      raw: Tháng 1, 1478
  [4/1500] ✓ gold='Tháng 1, 1562' extracted='Tháng 1, 1562' elapsed=1.05s
      Q: Giả sử bạn đang ở tháng 3, 1561, thời gian sau 10 tháng, thì là thời điểm nào?
      raw: Tháng 1, 1562
  [5/1500] ✓ gold='Tháng 4, 1129' extracted='Tháng 4

In [ ]:
# === EXP 2/6 — few_shot k=3 × vlsp_date (FT) ===
m = run_exp_ft('configs/few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

[runner] experiment=few_shot_vlsp_date_k3_ft method=few_shot dataset=vlsp_date verbose=True
[runner] loaded 1500 samples (task=date_arith, lang=vi)
[runner] reusing pre-loaded model: Qwen/Qwen3-8B (+adapter=/content/checkpoints/lora_vlsp_date_bf16)
[runner] few-shot k=3 for (date_arith,vi)
  [1/1500] ✓ gold='Tháng 4, 1321' extracted='Tháng 4, 1321' elapsed=1.11s
      Q: Giả sử bạn đang ở tháng 4, 1316, thời gian sau 4 năm 12 tháng, thì là thời điểm nào?
      raw: Tháng 4, 1321
  [2/1500] ✓ gold='Tháng 2, 1078' extracted='Tháng 2, 1078' elapsed=1.05s
      Q: Hãy tính thời điểm 10 năm trước tháng 2, 1088
      raw: Tháng 2, 1078
  [3/1500] ✓ gold='Tháng 1, 1478' extracted='Tháng 1, 1478' elapsed=1.06s
      Q: Ngày tháng nào sẽ là 1 năm 10 tháng trước tháng 11, 1479?
      raw: Tháng 1, 1478
  [4/1500] ✓ gold='Tháng 1, 1562' extracted='Tháng 1, 1562' elapsed=1.06s
      Q: Giả sử bạn đang ở tháng 3, 1561, thời gian sau 10 tháng, thì là thời điểm nào?
      raw: Tháng 1, 1562
  [5/1500

In [ ]:
# === EXP 3/6 — dynamic_few_shot k=3 × vlsp_date (FT)
# ⚠️ Data leakage: pool retrieval = rows 1500–2999 = train pool. Kết quả cần note. ===
m = run_exp_ft('configs/dynamic_few_shot_vlsp_date.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

[runner] experiment=dynamic_few_shot_vlsp_date_k3_ft method=dynamic_few_shot dataset=vlsp_date verbose=True
[runner] loaded 1500 samples (task=date_arith, lang=vi)
[runner] reusing pre-loaded model: Qwen/Qwen3-8B (+adapter=/content/checkpoints/lora_vlsp_date_bf16)
[runner] retrieval pool size=1500 (skip first 1500, exclude_eval_qids=False)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

[retrieval] building index: vlsp_date pool_size=1500 encoder=intfloat/multilingual-e5-base


/content/Temporal_Reasoning/src/retrieval/encoder.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

[retrieval] cached index -> outputs/retrieval_cache/vlsp_date-6b295dc3247f.faiss
[runner] dynamic few-shot k=3 balance_by_label=False unique_by_qid=False
  [1/1500] ✓ gold='Tháng 4, 1321' extracted='Tháng 4, 1321' elapsed=1.11s
      Q: Giả sử bạn đang ở tháng 4, 1316, thời gian sau 4 năm 12 tháng, thì là thời điểm nào?
      raw: Tháng 4, 1321
  [2/1500] ✓ gold='Tháng 2, 1078' extracted='Tháng 2, 1078' elapsed=1.09s
      Q: Hãy tính thời điểm 10 năm trước tháng 2, 1088
      raw: Tháng 2, 1078
  [3/1500] ✓ gold='Tháng 1, 1478' extracted='Tháng 1, 1478' elapsed=1.07s
      Q: Ngày tháng nào sẽ là 1 năm 10 tháng trước tháng 11, 1479?
      raw: Tháng 1, 1478
  [4/1500] ✓ gold='Tháng 1, 1562' extracted='Tháng 1, 1562' elapsed=1.07s
      Q: Giả sử bạn đang ở tháng 3, 1561, thời gian sau 10 tháng, thì là thời điểm nào?
      raw: Tháng 1, 1562
  [5/1500] ✓ gold='Tháng 4, 1129' extracted='Tháng 4, 1129' elapsed=1.08s
      Q: Thời gian 10 tháng trước tháng 2, 1130 là khi nào?
      raw: T

### `vlsp_duration` (cross-task, cùng VI) — kiểm tra catastrophic forgetting.

In [ ]:
# === EXP 4/6 — zero_shot × vlsp_duration (FT) ===
m = run_exp_ft('configs/zero_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

[runner] experiment=zero_shot_vlsp_duration_ft method=zero_shot dataset=vlsp_duration verbose=True
[runner] loaded 1500 samples (task=duration, lang=vi)
[runner] reusing pre-loaded model: Qwen/Qwen3-8B (+adapter=/content/checkpoints/lora_vlsp_date_bf16)
  [1/1500] ✓ gold='yes' extracted='yes' elapsed=0.22s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: yes
  [2/1500] ✓ gold='no' extracted='no' elapsed=0.20s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [3/1500] ✓ gold='yes' extracted='yes' elapsed=0.20s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: yes
  [4/1500] ✓ gold='no' extracted='no' elapsed=0.20s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [5/1500] ✓ gold='yes' extracted='yes' elapsed=0.20s
      Q: Mất bao lâu để tổ chức một sự kiện trực tuyến giới thiệu sản phẩm mới?
      raw: yes
  [100/1500] running: f1=0.807 p=0.772 r=0.846 

In [ ]:
# === EXP 5/6 — few_shot k=4 × vlsp_duration (FT) ===
m = run_exp_ft('configs/few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

[runner] experiment=few_shot_vlsp_duration_k4_ft method=few_shot dataset=vlsp_duration verbose=True
[runner] loaded 1500 samples (task=duration, lang=vi)
[runner] reusing pre-loaded model: Qwen/Qwen3-8B (+adapter=/content/checkpoints/lora_vlsp_date_bf16)
[runner] few-shot k=4 for (duration,vi)
  [1/1500] ✓ gold='yes' extracted='yes' elapsed=0.21s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: yes
  [2/1500] ✓ gold='no' extracted='no' elapsed=0.20s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [3/1500] ✓ gold='yes' extracted='yes' elapsed=0.20s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: yes
  [4/1500] ✓ gold='no' extracted='no' elapsed=0.20s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [5/1500] ✓ gold='yes' extracted='yes' elapsed=0.20s
      Q: Mất bao lâu để tổ chức một sự kiện trực tuyến giới thiệu sản phẩm mới?
      raw: yes
  [100

In [ ]:
# === EXP 6/6 — dynamic_few_shot k=4 × vlsp_duration (FT) ===
m = run_exp_ft('configs/dynamic_few_shot_vlsp_duration.yaml', verbose=True, verbose_every=200)
print(m['metrics'])

[runner] experiment=dynamic_few_shot_vlsp_duration_k4_ft method=dynamic_few_shot dataset=vlsp_duration verbose=True
[runner] loaded 1500 samples (task=duration, lang=vi)
[runner] reusing pre-loaded model: Qwen/Qwen3-8B (+adapter=/content/checkpoints/lora_vlsp_date_bf16)
[runner] retrieval pool size=4400 (skip first 1500, exclude_eval_qids=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[retrieval] building index: vlsp_duration pool_size=4400 encoder=intfloat/multilingual-e5-base


/content/Temporal_Reasoning/src/retrieval/encoder.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/138 [00:00<?, ?it/s]

[retrieval] cached index -> outputs/retrieval_cache/vlsp_duration-88a4c1f3a843.faiss
[runner] dynamic few-shot k=4 balance_by_label=True unique_by_qid=True
  [1/1500] ✗ gold='yes' extracted='no' elapsed=0.22s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [2/1500] ✓ gold='no' extracted='no' elapsed=0.22s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [3/1500] ✗ gold='yes' extracted='no' elapsed=0.22s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [4/1500] ✓ gold='no' extracted='no' elapsed=0.22s
      Q: Mất bao lâu để hoàn thành tất cả bài tập về nhà của lớp học?
      raw: no
  [5/1500] ✓ gold='yes' extracted='yes' elapsed=0.22s
      Q: Mất bao lâu để tổ chức một sự kiện trực tuyến giới thiệu sản phẩm mới?
      raw: yes
  [100/1500] running: f1=0.838 p=0.830 r=0.846 parse_fail=0 avg_time=0.219s
  [200/1500] ✓ gold='yes' extracted='yes'
  [200/1500] running: f1=0.817 

## So sánh baseline vs FT

In [ ]:
# === COMPARE — join baseline summary với FT summary trên (method, dataset) ===
import os
import pandas as pd

BASE_SUMMARY = '/content/Temporal_Reasoning/outputs/summary.csv' if not USE_DRIVE \
    else '/content/drive/MyDrive/Temporal_Reasoning/outputs/summary.csv'
FT_SUMMARY = os.path.join(DRIVE_OUT_FT, 'summary.csv')

def _load(path, label):
    if not os.path.exists(path):
        print(f'[warn] không tìm thấy {label} summary: {path}')
        return None
    df = pd.read_csv(path)
    keep = ['method', 'dataset', 'k_shot', 'metric', 'score', 'parse_fail']
    df = df[[c for c in keep if c in df.columns]].copy()
    df.columns = [c if c in ('method', 'dataset', 'k_shot', 'metric') else f'{c}_{label}' for c in df.columns]
    return df

base = _load(BASE_SUMMARY, 'base')
ft = _load(FT_SUMMARY, 'ft')
if base is not None and ft is not None:
    cmp = base.merge(ft, on=['method', 'dataset', 'k_shot', 'metric'], how='outer')
    cmp['delta'] = cmp['score_ft'] - cmp['score_base']
    print(cmp.sort_values(['dataset', 'method']).to_string(index=False))
else:
    print('Skip compare — thiếu summary file.')

ParserError: Error tokenizing data. C error: Expected 14 fields in line 5, saw 16


In [ ]:
# === EXPORT — zip outputs_ft để tải về máy trước khi Colab disconnect ===
import os
import shutil
from google.colab import files

os.chdir('/content')
out_dir = os.path.abspath(DRIVE_OUT_FT)
if not os.path.isdir(out_dir):
    raise FileNotFoundError(out_dir)
parent_dir = os.path.dirname(out_dir)
folder_name = os.path.basename(out_dir.rstrip('/'))
zip_path = shutil.make_archive(f'/content/{folder_name}', 'zip', root_dir=parent_dir, base_dir=folder_name)
print('Created zip:', zip_path)
files.download(zip_path)

Created zip: /content/outputs_ft.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import shutil
from google.colab import files

# Thư mục cần nén
out_dir = '/content/checkpoints'

# Kiểm tra tồn tại
if not os.path.isdir(out_dir):
    raise FileNotFoundError(out_dir)

# Lấy thông tin thư mục
parent_dir = os.path.dirname(out_dir)
folder_name = os.path.basename(out_dir.rstrip('/'))

# Tạo file zip
zip_path = shutil.make_archive(f'/content/{folder_name}', 'zip',
                              root_dir=parent_dir,
                              base_dir=folder_name)

print('Created zip:', zip_path)

# Tải về
files.download(zip_path)

Created zip: /content/checkpoints.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>